# Interactive Notebook 06 - IM simulation:

This interactive Jupyter notebook introduces basic simulation for induction machines.

For help with the installation of the required software, consider the comments in ```CTPD_course\interactive_notebooks\README.md```.
Throughout the exercises, we will be using a combination of scientific computation libraries from the [JAX](https://docs.jax.dev/en/latest/notebooks/thinking_in_jax.html) ecosystem and visualize them with [matplotlib](https://matplotlib.org/) and [ipywidgets](https://ipywidgets.readthedocs.io/en/stable/).

### Preliminaries & Imports:

In [ ]:
# automatically reloads imported ```.py```-files once they are changed and saved
%load_ext autoreload
%autoreload 2

In [ ]:
%%html
<style>
div.jupyter-widgets.widget-label {display: none;}
</style>

In [ ]:
# imports required packages
from copy import deepcopy
from functools import partial
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rc
mpl.rcParams.update({'font.size': 20})

import jax
import jax.numpy as jnp
import equinox as eqx
import diffrax

(**Optional**: If you have LaTeX installed, you can use the following lines for pretty rendering of plot labels.
Any LaTeX installation should work, as long as all the required packages are installed, e.g., [MiKTeX](https://miktex.org/) or [TeXLive](https://www.tug.org/texlive/).

If you do not have LaTeX installed, you can comment the next cell out or skip it.)

In [ ]:
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}\usepackage{xcolor}"

---

### Simulation of the linear (squirrel-cage) IM:

<div>
<img src="fig/IM_sim.png" width="1000"/>
</div>

The model can be written as (all quantities in the stator coordinate frame):

$$ \large
\begin{aligned}
    \boldsymbol{u}_{\mathrm{s, \upalpha\upbeta}} (t) &= R_\mathrm{s} \boldsymbol{i}_{\mathrm{s, \upalpha\upbeta}} (t) + \frac{\mathrm{d}}{\mathrm{d}t} \boldsymbol{\psi}_\mathrm{s, \upalpha\upbeta} (t) \\
    \boldsymbol{0} &= R_\mathrm{r} \boldsymbol{i}_{\mathrm{r, \upalpha\upbeta}} (t) - \omega_\mathrm{r, el} (t) \boldsymbol{J} \boldsymbol{\psi}_\mathrm{r, \upalpha\upbeta}(t) + \frac{\mathrm{d}}{\mathrm{d}t} \boldsymbol{\psi}_\mathrm{r, \upalpha\upbeta} (t) \\
    \boldsymbol{\psi}_\mathrm{s, \upalpha\upbeta} (t) &= (L_\mathrm{\sigma, s} + L_\mathrm{m}) \boldsymbol{i}_\mathrm{s, \upalpha\upbeta} (t) + L_\mathrm{m} \boldsymbol{i}_\mathrm{r, \upalpha\upbeta} (t) \\
    \boldsymbol{\psi}_\mathrm{r, \upalpha\upbeta} (t) &= (L_\mathrm{\sigma, r} + L_\mathrm{m}) \boldsymbol{i}_\mathrm{r, \upalpha\upbeta} (t) + L_\mathrm{m} \boldsymbol{i}_\mathrm{s, \upalpha\upbeta} (t) \\
    T(t) &= \frac{3}{2} p \left( \boldsymbol{i}_\mathrm{s, \upalpha\upbeta} (t) \right)^\top \boldsymbol{J}  \boldsymbol{\psi}_\mathrm{r, \upalpha\upbeta},\\
\end{aligned}
$$

with the parameter vector:

$$ \large
    \boldsymbol{\theta} = \begin{bmatrix} p & R_\mathrm{s} & R_\mathrm{r}  & L_\mathrm{\sigma, r} & L_\mathrm{\sigma, s} & L_\mathrm{m} \end{bmatrix}.
$$

In [ ]:
class StaticParams(eqx.Module):
    p: jax.Array = eqx.field(converter=jnp.asarray)  # Number of pole pairs
    R_s: jax.Array = eqx.field(converter=jnp.asarray)  # Stator resistance
    R_r: jax.Array = eqx.field(converter=jnp.asarray)  # Rotor resistance
    L_m: jax.Array = eqx.field(converter=jnp.asarray)
    L_sigs: jax.Array = eqx.field(converter=jnp.asarray)
    L_sigr: jax.Array = eqx.field(converter=jnp.asarray)
    i_lim: float
    u_dc: float

linear_motor_params = StaticParams(
    p=2,
    R_s=2.934,
    R_r=1.355,
    L_m=143.75e-3,
    L_sigs=5.87e-3,
    L_sigr=5.87e-3,
    i_lim=250,
    u_dc=560,
)

Due to the direct mapping between currents and flux based on the inductances, it is sufficient to consider only the stator currents and the rotor flux. This results in the system of ODEs:

$$ \large \begin{aligned}
    \frac{\mathrm{d}}{\mathrm{d}t} \boldsymbol{i}_{\mathrm{s, \upalpha\upbeta}} (t) &= - \frac{1}{\sigma L_\mathrm{s}}\left(R_\mathrm{s} + R_\mathrm{r} \frac{L_\mathrm{m}^2}{L_\mathrm{r}^2}\right)  \boldsymbol{i}_{\mathrm{s, \upalpha\upbeta}} (t) + \frac{L_\mathrm{m}}{\sigma L_\mathrm{s} L_\mathrm{r}} \left(\frac{R_\mathrm{r}}{L_\mathrm{r}} \mathbf{I} - \omega_{\mathrm{r,el}} \boldsymbol{J}\right) \boldsymbol{\psi}_\mathrm{r, \upalpha\upbeta} (t) + \frac{1}{\sigma L_\mathrm{s}} \boldsymbol{i}_{\mathrm{s, \upalpha\upbeta}} (t)\\
%
    \frac{\mathrm{d}}{\mathrm{d}t} \boldsymbol{\psi}_{\mathrm{r, \upalpha\upbeta}} (t) &= R_\mathrm{r} \frac{L_\mathrm{m}}{L_\mathrm{r}} \boldsymbol{i}_{\mathrm{s, \upalpha\upbeta}} (t) + \left(\omega_{\mathrm{r,el}} \boldsymbol{J} - \frac{R_\mathrm{r}}{L_\mathrm{r}} \mathbf{I} \right) \boldsymbol{\psi}_\mathrm{r, \upalpha\upbeta} (t)
\end{aligned}
$$
with $\large \boldsymbol{J} = \begin{bmatrix} 0 & -1 \\1 & 0\end{bmatrix}$, $\large \sigma = 1 - \frac{L_\mathrm{m}^2}{L_\mathrm{s}L_\mathrm{r}}$, $\large L_\mathrm{s} = L_\mathrm{m} + L_\mathrm{\sigma, s}$, and $\large L_\mathrm{r} = L_\mathrm{m} + L_\mathrm{\sigma, r}$.



Furthermore, we specifically distinguish the state $\boldsymbol{x}(t)$ from the observation $\boldsymbol{y} (t)$ and assume the rotation speed to be set externally and held constant:

$$ \large 
\begin{aligned}
    \boldsymbol{x} (t) &= \begin{bmatrix} i_\mathrm{s, \upalpha} (t) & i_\mathrm{s, \upbeta} (t) & \psi_\mathrm{r, \upalpha} (t) & \psi_\mathrm{r, \upbeta} (t) & \varepsilon_\mathrm{r} (t) & \omega_\mathrm{r, el} (t) \end{bmatrix}  \\
    & = \begin{bmatrix} i_\mathrm{s, \upalpha} (t) & i_\mathrm{s, \upbeta} (t) & \psi_\mathrm{r, \upalpha} (t) & \psi_\mathrm{r, \upbeta} (t) & \varepsilon_\mathrm{r} (t) & \omega_\mathrm{r, el} \end{bmatrix} \\
    \boldsymbol{y} (t) &= \begin{bmatrix} i_\mathrm{s, \upalpha} (t) & i_\mathrm{s, \upbeta} (t) &  \omega_\mathrm{r, el}\end{bmatrix}.
\end{aligned}
$$

The main idea is that, in practice, the rotor quantities cannot be measured and therefore are not directly available (except for the rotation speed $\omega_\mathrm{r, el}$). We still need to simulate them and, therefore, have knowledge of them for the simulation, but we assume to not have them from the outside for control purposes.

In [ ]:
class State(eqx.Module):
    i_s_alpha: jax.Array = eqx.field(converter=jnp.asarray)
    i_s_beta: jax.Array = eqx.field(converter=jnp.asarray)
    psi_r_alpha: jax.Array = eqx.field(converter=jnp.asarray)
    psi_r_beta: jax.Array = eqx.field(converter=jnp.asarray)
    epsilon_r: jax.Array = eqx.field(converter=jnp.asarray)
    omega_r_el: jax.Array = eqx.field(converter=jnp.asarray)

def generate_observation(state):
    return jnp.array([state.i_s_alpha, state.i_s_beta, state.omega_r_el])

In [ ]:
state = State(
    i_s_alpha=0.2,
    i_s_beta=0.15,
    psi_r_alpha=0.04,
    psi_r_beta=0.03,
    epsilon_r=0.0,
    omega_r_el=500,
)
eqx.tree_pprint(state, short_arrays=False)

generate_observation(state)

In [ ]:
def transpose_tree(list_of_states):
    return jax.tree.map(lambda *xs: jnp.array(xs), *list_of_states)



This motor model can then be programmatically expressed as an ODE:

In [ ]:
def linear_ode(state: State, u_s_alpha_beta: jax.Array, params: StaticParams):
    L_m = params.L_m
    L_s = L_m + params.L_sigs
    L_r = L_m + params.L_sigr
    R_s = params.R_s
    R_r = params.R_r

    J = jnp.array([[0, -1],[1, 0]])
    sigma = 1 - L_m**2 / (L_s * L_r)

    i_s_alpha_beta = jnp.array([state.i_s_alpha, state.i_s_beta])
    psi_r_alpha_beta = jnp.array([state.psi_r_alpha, state.psi_r_beta])
    
    di_dt = (
        - 1 / (sigma * L_s) * (R_s + R_r * (L_m**2 / L_r**2)) * i_s_alpha_beta
        + L_m / (sigma * L_s * L_r) * (R_r / L_r * jnp.eye(2) - state.omega_r_el * J) @ psi_r_alpha_beta
        + 1 / (sigma * L_s) * u_s_alpha_beta
    )

    dpsi_dt = (
        + R_r * (L_m / L_r) * i_s_alpha_beta
        + (state.omega_r_el * J - R_r / L_r * jnp.eye(2)) @ psi_r_alpha_beta
    )

    return di_dt, dpsi_dt

In [ ]:
linear_ode(state, jnp.array([34.02, 55.5]), linear_motor_params)

Simulation of the ODE with Euler is implemented as:

In [ ]:
@eqx.filter_jit
def euler_step(state: State, u_s_alpha_beta: jax.Array, params: StaticParams, T_s: float):
    di_dt, dpsi_dt = linear_ode(state, u_s_alpha_beta, params)

    next_i = jnp.array([state.i_s_alpha, state.i_s_beta]) + T_s * di_dt
    next_psi = jnp.array([state.psi_r_alpha, state.psi_r_beta]) + T_s * dpsi_dt
    next_eps = state.epsilon_r + T_s * state.omega_r_el

    new_state = eqx.tree_at(
        lambda s: (s.i_s_alpha, s.i_s_beta, s.psi_r_alpha, s.psi_r_beta, s.epsilon_r),
        state,
        (*next_i, *next_psi, next_eps)
    )
    obs = generate_observation(new_state)
    return obs, new_state

In [ ]:
T_s = 1e-4

obs, state = euler_step(state, jnp.array([34.02, 55.5]), linear_motor_params, T_s)
obs

In [ ]:
n = 4000

init_state = State(
    i_s_alpha=0.0,
    i_s_beta=0.0,
    psi_r_alpha=0.0,
    psi_r_beta=0.0,
    epsilon_r=0.0,
    omega_r_el=linear_motor_params.p * n * 2 * jnp.pi / 60,
)
init_obs = generate_observation(init_state)

In [ ]:
state = init_state

states = [init_state]
observations = [init_obs]

actions = jnp.ones((1000, 2)) * 15
actions = actions.at[500:, :].set(5.5)
for action in actions:
    obs, state = euler_step(state, action, linear_motor_params, T_s)

    states.append(state)
    observations.append(obs)

observations = jnp.array(observations)
states = transpose_tree(states)   # list of states to state of arrays

fig, ax = plt.subplots(figsize=(7, 4))
ax.grid(True, alpha=0.3)
ax.set_ylabel(r"$i_\mathrm{s}$ in $\mathrm{A}$")
ax.set_xlabel(r"$t$ in $\mathrm{s}$")
t = jnp.linspace(0, (observations.shape[0] - 1) * T_s, observations.shape[0])
ax.set_xlim((t[0], t[-1]))
ax.plot(t, states.i_s_alpha, label=r"$i_\mathrm{s, \upalpha}$")
ax.plot(t, states.i_s_beta, label=r"$i_\mathrm{s, \upbeta}$")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.grid(True, alpha=0.3)
ax.set_ylabel(r"$\psi_\mathrm{r}$ in $\mathrm{Vs}$")
ax.set_xlabel(r"$t$ in $\mathrm{s}$")
t = jnp.linspace(0, (observations.shape[0] - 1) * T_s, observations.shape[0])
ax.set_xlim((t[0], t[-1]))
ax.plot(t, states.psi_r_alpha, label=r"$\psi_\mathrm{r, \upalpha}$")
ax.plot(t, states.psi_r_beta, label=r"$\psi_\mathrm{r, \upbeta}$")
ax.legend()
plt.show()

### IM object:

From here on, we will once again switch to a object-based implementation of the induction machine simulator (code at `CTPD/interactive_notebooks/utils/IM/`)

In [ ]:
# auxiliary simulation code for the PMSM and the controller will is placed in `CTPD/interactive_notebooks/utils/`
import sys

from path_helper import get_folder_path
sys.path.insert(1, str(get_folder_path()))

In [ ]:
from utils.im.im import IM

In [ ]:
im = IM(
    T_s=T_s,
    solver=diffrax.Heun(),  # Explicit Euler is not stable enough
    saturated=True,
)

Exemplary saturation for $L_\mathrm{m}$:

In [ ]:
i_lim = im.env_properties.static_params.i_lim
i_s = jnp.linspace(-i_lim*2, i_lim*2, 1000)

l_m_sat = eqx.filter_vmap(im.get_L_saturated, in_axes=(0, None, None, None))(i_s, 0.0, 0.4, 0.4)

fig, ax = plt.subplots(figsize=(7, 4))
ax.grid(True, alpha=0.3)
ax.set_ylabel(r"$L_\mathrm{m}$")
ax.set_xlabel(r"$i_\mathrm{s,\upalpha}$")
ax.set_xlim((i_s[0], i_s[-1]))
ax.plot(i_s, l_m_sat)
plt.show()

In [ ]:
obs, state = im.reset()
obs, state = im.set_speed(n=4000, state=state)

states = [state]
observations = [obs]

actions = jnp.ones((1000, 2)) * 15
actions = actions.at[500:, :].set(5.5)

for action in actions:
    obs, state = im.step(state, action)

    states.append(state)
    observations.append(obs)

observations = jnp.array(observations)
states = transpose_tree(states)   # list of states to state of arrays

fig, ax = plt.subplots(figsize=(7, 4))
ax.grid(True, alpha=0.3)
ax.set_ylabel(r"$i_\mathrm{s}$ in $\mathrm{A}$")
ax.set_xlabel(r"$t$ in $\mathrm{s}$")
t = jnp.linspace(0, (observations.shape[0] - 1) * T_s, observations.shape[0])
ax.set_xlim((t[0], t[-1]))
ax.plot(t, states.physical_state.i_s_alpha, label=r"$i_\mathrm{s, \upalpha}$")
ax.plot(t, states.physical_state.i_s_beta, label=r"$i_\mathrm{s, \upbeta}$")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.grid(True, alpha=0.3)
ax.set_ylabel(r"$\psi_\mathrm{r}$ in $\mathrm{Vs}$")
ax.set_xlabel(r"$t$ in $\mathrm{s}$")
t = jnp.linspace(0, (observations.shape[0] - 1) * T_s, observations.shape[0])
ax.set_xlim((t[0], t[-1]))
ax.plot(t, states.physical_state.psi_r_alpha, label=r"$\psi_\mathrm{r, \upalpha}$")
ax.plot(t, states.physical_state.psi_r_beta, label=r"$\psi_\mathrm{r, \upbeta}$")
ax.legend()
plt.show()

Note that the simulation is now running in the $\upalpha\upbeta$-coordinate frame. 
For the control task, it is highly beneficial to consider the quanitities in the rotor-oriented $\mathrm{dq}$-coordinate frame. 

For the simulation, we can transform the currents and flux into the $\mathrm{dq}$-frame (which is not directly possible in practice, since we cannot measure the rotor flux angle):

In [ ]:
i_s_alpha = states.physical_state.i_s_alpha
i_s_beta = states.physical_state.i_s_beta
eps_r = states.physical_state.epsilon

cos_e, sin_e = jnp.cos(eps_r), jnp.sin(eps_r)
i_s_d = i_s_alpha * cos_e + i_s_beta * sin_e
i_s_q = -i_s_alpha * sin_e + i_s_beta * cos_e


fig, ax = plt.subplots(figsize=(7, 4))
ax.grid(True, alpha=0.3)
ax.set_ylabel(r"$i_\mathrm{s}$ in $\mathrm{A}$")
ax.set_xlabel(r"$t$ in $\mathrm{s}$")
t = jnp.linspace(0, (observations.shape[0] - 1) * T_s, observations.shape[0])
ax.set_xlim((t[0], t[-1]))
ax.plot(t, i_s_d, label=r"$i_\mathrm{s, d}$")
ax.plot(t, i_s_q, label=r"$i_\mathrm{s, q}$")
ax.legend()
plt.show()

In [ ]:
torques = eqx.filter_vmap(im.currents_to_torque_sat)(
    states.physical_state.i_s_alpha,
    states.physical_state.i_s_beta,
    states.physical_state.psi_r_alpha,
    states.physical_state.psi_r_beta,
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.grid(True, alpha=0.3)
ax.set_ylabel(r"$T$ in $\mathrm{Nm}$")
ax.set_xlabel(r"$t$ in $\mathrm{s}$")
t = jnp.linspace(0, (observations.shape[0] - 1) * T_s, observations.shape[0])
ax.set_xlim((t[0], t[-1]))
ax.plot(t, torques)
plt.show()

We currently apply a constant voltage in $\upalpha\upbeta$ which results in osscillating currents in $\mathrm{dq}$.
We would much rather have oscillating behavior in $\upalpha\upbeta$ and constant currents in $\mathrm{dq}$.
It is generally much simpler to control the system in $\mathrm{dq}$, thereby, we want to find ways to construct an approximated $\mathrm{dq}$-coordinate frame in the following. 

### Current model observer:

<div>
<img src="fig/IM_observer.png" width="1000"/>
</div>

As the rotor flux is not directly measurable, it needs to be estimated from the measurable quantities.
An established method is the current model observer (CMO). There, the rotor flux angle, slip frequency, and rotor flux magnitude are estimated based on the rotor angle, stator currents and rotation speed of the machine.
For this we are considering a rotor-oriented $\mathrm{dq}$-frame for our quantities (see pp. 322-323 in the slides).
This means that our quantities are transformed from $\upalpha\upbeta$ with
$T_\mathrm{p} (\varepsilon_{\psi_\mathrm{r}})$
to the rotor-oriented $\mathrm{dq}$-frame, where $\varepsilon_{\psi_\mathrm{r}}$ is the angle of the rotor flux.

By definition, the $\mathrm{q}$-component of the rotor flux is $0$, as the vector is aligned with the $\mathrm{d}$-axis, i.e.,
$$\large \boldsymbol{\psi}_{\mathrm{r, dq}}^{\psi_\mathrm{r}} (t) = \begin{bmatrix} \psi_{\mathrm{r, d}}^{\psi_\mathrm{r}} (t) \\ \psi_{\mathrm{r, q}}^{\psi_\mathrm{r}} (t)\end{bmatrix} = \begin{bmatrix} \psi_{\mathrm{r, d}}^{\psi_\mathrm{r}} (t) \\ 0 \end{bmatrix} = \begin{bmatrix} \| \boldsymbol{\psi}_{\mathrm{r, dq}}^{\psi_\mathrm{r}} (t) \|  \\ 0 \end{bmatrix}$$

The propagation of the flux can then be defined by 
$$\large \frac{\mathrm{d}}{\mathrm{d}t} \psi_{\mathrm{r, d}}^{\psi_\mathrm{r}} (t) = - R_\mathrm{r} i_\mathrm{r,d}^{\psi_\mathrm{r}} (t) + \left( \omega_{\psi_{\mathrm{r}}} (t) - \omega (t) \right) \underbrace{\psi_{\mathrm{r, q}}^{\psi_\mathrm{r}} (t)}_{= \, 0, \text{ by def.}} =  - R_\mathrm{r} i_\mathrm{r,d}^{\psi_\mathrm{r}} (t)$$
Through insertion of 
$$\large
\begin{aligned}
    \psi^{\psi_\mathrm{r}}_\mathrm{r, d} (t) &= L_\mathrm{r} i^{\psi_\mathrm{r}}_\mathrm{r, d} (t) + L_\mathrm{m} i^{\psi_\mathrm{r}}_\mathrm{s, d} (t) \\
    \iff i^{\psi_\mathrm{r}}_\mathrm{r, d} (t) &= \frac{1}{L_\mathrm{r}} \psi^{\psi_\mathrm{r}}_\mathrm{r, d} (t) - \frac{L_\mathrm{m}}{L_\mathrm{r}} i^{\psi_\mathrm{r}}_\mathrm{s, d} (t)
\end{aligned}
$$
it follows that
$$\large
\begin{aligned}
    \frac{\mathrm{d}}{\mathrm{d}t} \psi_{\mathrm{r, d}}^{\psi_\mathrm{r}} (t) = - \frac{R_\mathrm{r}}{L_\mathrm{r}} \psi^{\psi_\mathrm{r}}_\mathrm{r, d} (t) + \frac{R_\mathrm{r}L_\mathrm{m}}{L_\mathrm{r}} i^{\psi_\mathrm{r}}_\mathrm{s, d} (t).
\end{aligned}
$$

This equation can then be used to propagate an estimate $\hat{\psi}_{\mathrm{r, d}}^{\psi_\mathrm{r}} (t)$ of the flux, e.g. with explicit Euler:
$$\large
\begin{aligned}
    \hat{\psi}_{\mathrm{r, d}}^{\psi_\mathrm{r}} [k+1] = \hat{\psi}_{\mathrm{r, d}}^{\psi_\mathrm{r}} [k] + T_\mathrm{s} \cdot \left(- \frac{R_\mathrm{r}}{L_\mathrm{r}} \hat{\psi}^{\psi_\mathrm{r}}_\mathrm{r, d} [k] + \frac{R_\mathrm{r}L_\mathrm{m}}{L_\mathrm{r}} i^{\psi_\mathrm{r}}_\mathrm{s, d} [k] \right).
\end{aligned}
$$

Additionally, we require an estimate of the rotor-flux angle to be able to transform quantities into the rotor-oriented $\mathrm{dq}$-frame (see p. 323 in the slides).
The change of $\varepsilon_{\psi_\mathrm{r}}$ is the sum of the electrical rotor frequency and the slip frequency. 
$$\large
\begin{aligned}
    \hat{\omega}_\mathrm{slip} (t) = \frac{L_\mathrm{m} R_\mathrm{r}}{L_\mathrm{r}} \frac{i_\mathrm{s,q}^{\psi_\mathrm{r}}(t)}{\psi_\mathrm{r,d}^{\psi_\mathrm{r}}(t)}
\end{aligned}
$$
the propagation of $\varepsilon_{\psi_\mathrm{r}}$ can be estimated as

$$\large
\begin{aligned}
    \hat{\varepsilon}_{\psi_\mathrm{r}} [k+1] &= \hat{\varepsilon}_{\psi_\mathrm{r}} [k] + T_\mathrm{s} \cdot \left( \omega_\mathrm{r,el} [k] + \hat{\omega}_\mathrm{slip} [k] \right) \\
                                              &= \hat{\varepsilon}_{\psi_\mathrm{r}} [k] + T_\mathrm{s} \cdot \left( \omega_\mathrm{r,el} [k] + \frac{L_\mathrm{m} R_\mathrm{r}}{L_\mathrm{r}} \frac{i_\mathrm{s,q}^{\psi_\mathrm{r}}(t)}{\psi_\mathrm{r,d}^{\psi_\mathrm{r}}(t)} \right)
\end{aligned}
$$

It should be noted, that we require an accurate initial guess for $\hat{\psi}_{\mathrm{r, d}}^{\psi_\mathrm{r}} [k=0]$ and $\hat{\varepsilon}_{\psi_\mathrm{r}} [k=0]$ and that all errors in the estimate are cumulated throughout a simulation, since there is no feedback in this observer.
There are more sophisticated observer structures that address these issues, but for the sake of this tutorial, we will focus on the CMO.

In [ ]:
class CMO(eqx.Module):

    R_r: float
    L_sigr: float
    L_m: float
    p: int
    T_s: float

    class CMOState(eqx.Module):
        eps_r_hat: jax.Array = eqx.field(converter=jnp.asarray)
        psi_r_hat: jax.Array = eqx.field(converter=jnp.asarray)
        omega_psi_rs_hat: jax.Array = eqx.field(converter=jnp.asarray)

    def __init__(self, R_r, L_sigr, L_m, p, T_s):
        self.R_r = R_r
        self.L_sigr = L_sigr
        self.L_m = L_m
        self.p = p
        self.T_s = T_s

    def reset(
        self,
        eps_r_hat=None,
        psi_r_hat=None,
        omega_psi_rs_hat=None,
    ):
        if eps_r_hat is None:
            eps_r_hat = 0.0
        if psi_r_hat is None:
            psi_r_hat = 0.0
        if omega_psi_rs_hat is None:
            omega_psi_rs_hat = 0.0

        return self.CMOState(
            eps_r_hat=eps_r_hat,
            psi_r_hat=psi_r_hat,
            omega_psi_rs_hat=omega_psi_rs_hat,
        )

    @eqx.filter_jit
    def __call__(
        self,
        state: CMOState,
        i_s_alpha: float,
        i_s_beta: float,
        omega_rs: float,        
    ):
        eps_r_hat = state.eps_r_hat
        psi_r_hat = state.psi_r_hat

        # alpha-beta to dq
        cos_e, sin_e = jnp.cos(eps_r_hat), jnp.sin(eps_r_hat)
        i_s_d = i_s_alpha * cos_e + i_s_beta * sin_e
        i_s_q = -i_s_alpha * sin_e + i_s_beta * cos_e

        # predict next flux
        L_r = self.L_sigr + self.L_m
        psi_r_hat_next = psi_r_hat + self.T_s * (-self.R_r / L_r * psi_r_hat + self.R_r * self.L_m / L_r * i_s_d)

        # predict next angle
        omega_slip_hat = self.R_r * self.L_m / (L_r * (psi_r_hat_next + 1e-8)) * i_s_q
        omega_psi_rs_hat = omega_slip_hat + omega_rs
        eps_r_hat_next = eps_r_hat + self.T_s * omega_psi_rs_hat

        new_state = self.CMOState(
            psi_r_hat=psi_r_hat_next,
            eps_r_hat=eps_r_hat_next,
            omega_psi_rs_hat=omega_psi_rs_hat,
        )

        return new_state, omega_psi_rs_hat

In [ ]:
cmo = CMO(
    R_r=linear_motor_params.R_r,
    L_sigr=linear_motor_params.L_sigr,
    L_m=linear_motor_params.L_m,
    p=linear_motor_params.p,
    T_s=T_s,
)

In [ ]:
cmo_state = cmo.reset()
eqx.tree_pprint(cmo_state, short_arrays=False)

In [ ]:
obs, state = im.reset()
obs, state = im.set_speed(n=4000, state=state)

cmo_state = cmo.reset()

cmo_states = [cmo_state]

states = [state]
observations = [obs]

actions = jnp.ones((1000, 2)) * 15
actions = actions.at[500:, :].set(5.5)

for action in actions:
    obs, state = im.step(state, action)

    cmo_state, _ = cmo(
        cmo_state,
        state.physical_state.i_s_alpha,
        state.physical_state.i_s_beta,
        state.physical_state.omega_el
    )

    states.append(state)
    observations.append(obs)
    cmo_states.append(cmo_state)

observations = jnp.array(observations)
states = transpose_tree(states)   # list of states to state of arrays
cmo_states = transpose_tree(cmo_states)  # list of states to state of arrays

In [ ]:
raise NotImplementedError("Transform fluxes back to dq, show in dq plot")
raise NotADirectoryError("Show dq-transformation of currents next to true values")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.grid(True, alpha=0.3)
ax.set_ylabel(r"$i_\mathrm{s}$ in $\mathrm{A}$")
ax.set_xlabel(r"$t$ in $\mathrm{s}$")
t = jnp.linspace(0, (observations.shape[0] - 1) * T_s, observations.shape[0])
ax.set_xlim((t[0], t[-1]))
ax.plot(t, states.physical_state.i_s_alpha, label=r"$i_\mathrm{s, \upalpha}$")
ax.plot(t, states.physical_state.i_s_beta, label=r"$i_\mathrm{s, \upbeta}$")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.grid(True, alpha=0.3)
ax.set_ylabel(r"$\psi_\mathrm{r}$ in $\mathrm{Vs}$")
ax.set_xlabel(r"$t$ in $\mathrm{s}$")
t = jnp.linspace(0, (observations.shape[0] - 1) * T_s, observations.shape[0])
ax.set_xlim((t[0], t[-1]))
ax.plot(t, states.physical_state.psi_r_alpha, label=r"$\psi_\mathrm{r, \upalpha}$")
ax.plot(t, states.physical_state.psi_r_beta, label=r"$\psi_\mathrm{r, \upbeta}$")
ax.legend()
plt.show()

### Current and flux control:

<div>
<img src="fig/IM_control.png" width="1000"/>
</div>

For the controller, we utilize the $\mathrm{dq}$-coordinate frame.